In [13]:
#setup + imports

import sys
sys.path.append('..')

import pandas as pd
import os
import re
from config import engine

# Helper function to extract uid from filename
def extract_uid(filename):
    """Extract uid from filenames like 'activity_u01.csv' -> 'u01'"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")

✓ Setup complete


In [18]:
# ===========================
# EDUCATION TABLES (CORRECTED)
# ===========================

education_dir = '/Users/test/Downloads/Data Management & Databases/studentlife_project/data/education'

# ----------------------------
# 1. GRADES
# ----------------------------
def load_grades():
    """Load student grades"""
    filepath = os.path.join(education_dir, 'grades.csv')
    
    df = pd.read_csv(filepath)
    
    # Fix column names (remove leading spaces)
    df.columns = df.columns.str.strip()
    
    # Rename to match YOUR schema
    df = df.rename(columns={
        'gpa all': 'cumulative_gpa',
        'gpa 13s': 'term_gpa',
        'cs 65': 'course_grade'
    })
    
    # Don't include grade_id - it will auto-increment
    df.to_sql('grade', engine, if_exists='append', index=False)
    
    print(f"✓ Loaded {len(df):,} grade records")
    return len(df)

# ----------------------------
# 2. DEADLINES
# ----------------------------
def load_deadlines():
    """Load assignment deadlines - transform from wide to long format"""
    filepath = os.path.join(education_dir, 'deadlines.csv')
    
    df = pd.read_csv(filepath)
    
    # Melt from wide to long format
    df_long = df.melt(
        id_vars=['uid'],
        var_name='date',  # Changed from 'deadline_date'
        value_name='count'  # Changed from 'num_deadlines'
    )
    
    # Filter out rows where count is 0 or NaN
    df_long = df_long[df_long['count'] > 0].copy()
    
    # Convert date string to proper date format
    df_long['date'] = pd.to_datetime(df_long['date'])
    
    # Don't include deadline_id - it will auto-increment
    df_long.to_sql('deadline', engine, if_exists='append', index=False)
    
    print(f"✓ Loaded {len(df_long):,} deadline records")
    return len(df_long)

# ----------------------------
# 3. PIAZZA_USAGE
# ----------------------------
def load_piazza():
    """Load Piazza activity"""
    filepath = os.path.join(education_dir, 'piazza.csv')
    
    df = pd.read_csv(filepath)
    
    # Check what columns your schema expects
    print("\nPIAZZA_USAGE table columns:")
    result = pd.read_sql("SELECT * FROM piazza_usage LIMIT 0", engine)
    print(result.columns.tolist())
    
    # Rename columns to match schema (remove spaces)
    df = df.rename(columns={
        'days online': 'days_online'
    })
    
    df.to_sql('piazza_usage', engine, if_exists='append', index=False)
    
    print(f"✓ Loaded {len(df):,} Piazza records")
    return len(df)

# ----------------------------
# RUN ALL
# ----------------------------
print("="*60)
print("LOADING EDUCATION TABLES (CORRECTED)")
print("="*60 + "\n")

try:
    load_grades()
except Exception as e:
    print(f"❌ Error loading grades: {e}")

try:
    load_deadlines()
except Exception as e:
    print(f"❌ Error loading deadlines: {e}")

try:
    load_piazza()
except Exception as e:
    print(f"❌ Error loading piazza: {e}")

print("\n" + "="*60)
print("EDUCATION TABLES COMPLETE")
print("="*60)

LOADING EDUCATION TABLES (CORRECTED)

✓ Loaded 30 grade records
✓ Loaded 644 deadline records

PIAZZA_USAGE table columns:
['uid', 'days_online', 'views', 'contributions', 'questions', 'notes', 'answers']
❌ Error loading piazza: Execution failed on sql 'INSERT INTO piazza_usage (uid, days_online, views, contributions, questions, notes, answers) VALUES (:uid, :days_online, :views, :contributions, :questions, :notes, :answers)': (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "piazza_usage_pkey"
DETAIL:  Key (uid)=(u00) already exists.

[SQL: INSERT INTO piazza_usage (uid, days_online, views, contributions, questions, notes, answers) VALUES (%(uid__0)s, %(days_online__0)s, %(views__0)s, %(contributions__0)s, %(questions__0)s, %(notes__0)s, %(answers__0)s), (%(uid__1)s, %(days_online__1)s, ... 5803 characters truncated ... line__48)s, %(views__48)s, %(contributions__48)s, %(questions__48)s, %(notes__48)s, %(answers__48)s)]
[parameters: {'contributions__0':

In [ ]:
# Check piazza status (may have already been loaded)
result = pd.read_sql("SELECT COUNT(*) as count FROM piazza_usage", engine)
print(f"Piazza records already in database: {result['count'][0]}")

Piazza records already in database: 49
